### TEST · dominio pequeño (Mediterráneo occidental, 30–48 °N, 5 °W–20 °E)
¿Concentrar las 30 PCs en la región donde se genera el oleaje mejora el XGB en extremos? Todo se guarda con el sufijo `_wmed`; el pipeline de dominio completo no se toca.

In [16]:
import os, glob

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from bluemath_tk.datamining import PCA
from bluemath_tk.core.io import load_model

from sklearn.decomposition import PCA as PCA_
from bluemath_tk.core.operations import spatial_gradient
from joblib import Parallel, delayed

#### Load Data

In [17]:
pcs_df = pd.read_csv('outputs/pca/pcs_msl_30_6h_wmed.csv', index_col=0, parse_dates=True)
cols = list(pcs_df.columns)

pca_msl = load_model('outputs/pca/pca_msl_wmed.pkl')

era5 = xr.open_zarr('data/era5_mslp_grad_1985_2026_1deg_hourly.zarr').rename({'valid_time': 'time'})
era5 = era5.sel(time=era5.time.dt.hour.isin([0, 6, 12, 18]))
era5 = era5.sel(latitude=slice(48, 30), longitude=slice(-5, 20))                     # dominio pequeño

training = slice('1985-01-01', '2015-12-31')

sim_files = sorted(glob.glob('outputs/var_6h/simulaciones_h1/sim_*.parquet'))
print(len(sim_files))

100


#### PCA del gradiente real (bluemath)

In [18]:
pca_grad = PCA(n_components=50)
pca_grad._pca = PCA_(n_components=50, svd_solver='covariance_eigh')
pca_grad.fit_transform(data=era5[['grad2']].sel(time=training), vars_to_stack=['grad2'],
                       coords_to_stack=['latitude', 'longitude'], pca_dim_for_rows='time')

cum = np.cumsum(pca_grad.explained_variance_ratio)
print(f'10 PCs: {cum[9]:.3f} | 50 PCs: {cum[49]:.3f}')

2026-11-26 13:42:43,708 - PCA - WARNING - Using PCA
2026-11-26 13:42:45,166 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method


10 PCs: 0.587 | 50 PCs: 0.872


#### PCs de presión → campo (inverse_transform) → gradiente → PCs del gradiente

In [19]:
def grad_pcs(pcs, n_gr=10, paso=8000):
    out = []
    for i in range(0, len(pcs), paso):
        p = pcs.iloc[i:i + paso][cols]
        pcs_da = xr.DataArray(p.values, dims=('time', 'n_component'),
                              coords={'time': p.index, 'n_component': np.arange(len(cols))})
        campo = pca_msl.inverse_transform(pcs_da)['msl'].transpose('time', 'latitude', 'longitude')
        grad = spatial_gradient(campo).to_dataset(name='grad2')
        out.append(pca_grad.transform(data=grad)['PCs'].values[:, :n_gr])
    return pd.DataFrame(np.concatenate(out), index=pcs.index, columns=[f'grec_PC{k + 1}' for k in range(n_gr)])

#### Histórico

In [20]:
grec_hist = grad_pcs(pcs_df)

#### Comprobación: reconstruido vs real en training
grec_real = pca_grad.transform(data=era5[['grad2']].sel(time=training))['PCs'].values[:, :10]
t_train = grec_hist.loc[training].index
print(pd.Series([np.corrcoef(grec_real[:, k], grec_hist.loc[t_train].values[:, k])[0, 1] for k in range(10)],
                index=grec_hist.columns).round(2))

2026-11-26 13:42:47,799 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 13:42:50,015 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 13:42:52,283 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 13:42:54,549 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 13:42:56,816 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 13:42:59,084 - PCA - WARNING - Using 450 out of 450 av

grec_PC1     0.99
grec_PC2     0.97
grec_PC3     0.97
grec_PC4     0.96
grec_PC5     0.93
grec_PC6     0.95
grec_PC7     0.91
grec_PC8     0.90
grec_PC9     0.83
grec_PC10    0.85
dtype: float64


In [21]:
os.makedirs('outputs/grad_rec', exist_ok=True)
grec_hist.to_csv('outputs/grad_rec/pcs_grad_rec_6h_wmed.csv', float_format='%.5f')
pca_grad.save_model('outputs/grad_rec/pca_grad_wmed.pkl')

2026-11-26 13:43:24,111 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!


#### Simulaciones h1 del PVAR (dominio pequeño)

#### Gradiente de las simulaciones de las variantes de prueba (h1b, h2b, e4b…)

In [22]:
from joblib import Parallel, delayed

VARIANTES_GRAD = ['h1b', 'h2b', 'e4b', 'm12b']          # las que ya están hechas se saltan

def procesar(f, out_dir):
    f_out = os.path.join(out_dir, os.path.basename(f).replace('sim_', 'grec_'))
    if not os.path.exists(f_out):
        grad_pcs(pd.read_parquet(f)).astype('float32').to_parquet(f_out)

for nombre in VARIANTES_GRAD:
    sim_files = sorted(glob.glob(f'outputs/var_6h_wmed/simulaciones_{nombre}/sim_*.parquet'))
    out_dir = f'outputs/grad_rec/simulaciones_{nombre}_wmed'
    os.makedirs(out_dir, exist_ok=True)
    print(nombre, len(sim_files))
    _ = Parallel(n_jobs=os.cpu_count() - 2, verbose=0)(delayed(procesar)(f, out_dir) for f in sim_files)

2026-11-26 13:43:24,143 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:24,153 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!


h1b 20


2026-11-26 13:43:24,614 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:24,618 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:24,621 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:24,629 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:24,635 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:24,638 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:24,645 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:24,647 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:24,652 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26

h2b 20


2026-11-26 13:43:35,848 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:35,863 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:35,877 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:35,879 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:35,883 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:35,885 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:35,889 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:35,891 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:35,894 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26

e4b 20


2026-11-26 13:43:37,021 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:37,028 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:37,043 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:37,044 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:37,049 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:37,050 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:37,054 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:37,056 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 13:43:37,060 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26

m12b 20
